# AI Smart Electricity System — Bangladesh

**Task:** predict next month electricity usage, estimate the bill in BDT, and detect anomalies.
**Models:** Linear Regression, SVM, Random Forest, Decision Tree.
**Dataset:** Dhaka daily electricity demand + temperature (2021–2025, Kaggle).

## 1. Load and clean the data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# dataset file: Kaggle input path, or local folder
PATH = "/kaggle/input/dhaka-2021-25-electricity-and-temperature-data/Updated_Final_Dataset_2021_2025.xlsx"
if not os.path.exists(PATH):
    PATH = "data/Updated_Final_Dataset_2021_2025.xlsx"

# read, keep useful columns, drop rows without demand
df = pd.read_excel(PATH, parse_dates=["Date"])
df = df[["Date", "Demand (MW)", "Temp (°C)", "Holiday"]].dropna().sort_values("Date")
df.columns = ["date", "demand_mw", "temp_c", "holiday"]
df = df.reset_index(drop=True)
print(df.shape)
df.tail(3)

## 2. Make simple features

In [ ]:
d = df.set_index("date")["demand_mw"]

# features: calendar + weather + recent demand
F = pd.DataFrame(index=d.index)
F["temp"] = df.set_index("date")["temp_c"]
F["holiday"] = df.set_index("date")["holiday"].astype(int)
F["month"] = d.index.month
F["dayofweek"] = d.index.dayofweek
F["is_weekend"] = (d.index.dayofweek >= 5).astype(int)
F["lag_1"] = d.shift(1)          # yesterday
F["lag_7"] = d.shift(7)          # same day last week
F["roll_7"] = d.shift(1).rolling(7).mean()  # 7-day average
F["demand"] = d
F = F.dropna()

FEATURES = ["temp", "holiday", "month", "dayofweek", "is_weekend", "lag_1", "lag_7", "roll_7"]
F.head(3)

## 3. Train / test split (time based)

In [ ]:
# train = 2021-2024, test = 2025
train = F[F.index <= "2024-12-31"]
test = F[F.index >= "2025-01-01"]
X_train, y_train = train[FEATURES], train["demand"]
X_test, y_test = test[FEATURES], test["demand"]
print("train:", len(train), "| test:", len(test))

## 4. Train the four models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Linear Regression": LinearRegression(),
    # SVM needs scaled inputs, so we wrap it in a small pipeline
    "SVM": Pipeline([("scale", StandardScaler()),
                     ("svr", SVR(kernel="rbf", C=100))]),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print("trained:", name)

## 5. Compare the models

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rows, preds = [], {}
for name, m in models.items():
    p = preds[name] = pd.Series(m.predict(X_test), index=X_test.index)
    rows.append({
        "model": name,
        "RMSE": round(np.sqrt(mean_squared_error(y_test, p)), 1),
        "MAE": round(mean_absolute_error(y_test, p), 1),
        "MAPE (%)": round(np.mean(np.abs((y_test - p) / y_test)) * 100, 2),
        "R2": round(r2_score(y_test, p), 4),
    })

metrics = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
best_name = metrics.iloc[0]["model"]
best_model = models[best_name]
print("BEST MODEL:", best_name)
metrics

In [ ]:
# lower RMSE = better
plt.figure(figsize=(7, 3.5))
plt.bar(metrics["model"], metrics["RMSE"], color="#0D9488")
plt.title("Test RMSE by model (MW)")
plt.ylabel("RMSE (MW)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Predict next month

In [ ]:
# forecast the days after the data ends, keep the target month (November 2025)
hist = d.copy()
last_day = hist.index.max()
next_month = pd.date_range("2025-11-01", "2025-11-30", freq="D")

# future temperature = average November temperature of past years
nov_temp = df[df.date.dt.month == 11]["temp_c"].mean()

cur = hist.copy()
days = []
for ts in pd.date_range(last_day + pd.Timedelta(days=1), next_month[-1], freq="D"):
    row = pd.DataFrame([{
        "temp": nov_temp, "holiday": 0,
        "month": ts.month, "dayofweek": ts.dayofweek,
        "is_weekend": int(ts.dayofweek >= 5),
        "lag_1": cur.iloc[-1], "lag_7": cur.iloc[-7], "roll_7": cur.iloc[-7:].mean(),
    }], columns=FEATURES)
    p = float(best_model.predict(row)[0])
    cur.loc[ts] = p                      # feed the prediction back for the next day
    days.append({"date": ts, "demand_mw": p})

fcast = pd.DataFrame(days).set_index("date")
fmonth = fcast.loc[next_month[0]:next_month[-1], "demand_mw"]

In [ ]:
# summary of the next month
total_gwh = fmonth.sum() * 24 / 1000
avg_mw = fmonth.mean()
peak_day = fmonth.idxmax()
last_nov = d.loc["2024-11-01":"2024-11-30"]
yoy = (fmonth.sum() / last_nov.sum() - 1) * 100

print(f"Next month   : {next_month[0]:%B %Y}")
print(f"Total usage  : {total_gwh:,.0f} GWh")
print(f"Average load : {avg_mw:,.0f} MW")
print(f"Peak day     : {peak_day:%d %b} ({fmonth.max():,.0f} MW)")
print(f"vs Nov 2024  : {yoy:+.1f}%")

In [ ]:
# chart: last 60 real days + next month forecast
rmse = float(metrics.iloc[0]["RMSE"])
plt.figure(figsize=(10, 3.8))
hist.tail(60).plot(label="actual", color="#334155")
fmonth.index = fmonth.index
plt.plot(fmonth.index, fmonth.values, label="forecast", color="#C05F0A", linewidth=2)
plt.fill_between(fmonth.index, fmonth - rmse, fmonth + rmse,
                 color="#C05F0A", alpha=0.15, label="± RMSE")
plt.title(f"Next month forecast — {best_name}")
plt.ylabel("Demand (MW)")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Estimate the bill in BDT

In [ ]:
# Bangladesh residential tariff (BERC 2026, category LT-A)
SLABS = [(75, 5.26), (200, 8.50), (300, 9.10), (400, 9.62), (600, 15.01), (99999, 17.35)]
LIFELINE_RATE, LIFELINE_MAX = 4.63, 50   # only if monthly use <= 50 units
DEMAND_CHARGE, VAT = 42.0, 0.05          # 42 BDT per kW load, 5% VAT

def bill_bdt(units, kw=1.0):
    # energy charge slab by slab
    if units <= LIFELINE_MAX:
        energy = units * LIFELINE_RATE
    else:
        energy, prev = 0.0, 0
        for limit, rate in SLABS:
            energy += max(0, min(units, limit) - prev) * rate
            prev = limit
    demand = DEMAND_CHARGE * kw
    vat = (energy + demand) * VAT
    return {"units": units, "energy": round(energy, 2),
            "demand": round(demand, 2), "vat": round(vat, 2),
            "total": round(energy + demand + vat, 2)}

# example bills for typical monthly usage
pd.DataFrame([bill_bdt(u) for u in [50, 100, 150, 200, 250, 300, 400, 600]])

## 8. Detect anomalies

In [ ]:
# anomaly = test-day error far from normal (residual z-score beyond ±3)
best_pred = preds[best_name]
resid = y_test - best_pred
z = (resid - resid.mean()) / resid.std()
anom = z[abs(z) > 3].sort_values(key=abs, ascending=False)

print(f"Anomalies found: {len(anom)}")
out = pd.DataFrame({
    "date": anom.index.date,
    "actual_mw": y_test[anom.index].round(0).values,
    "expected_mw": best_pred[anom.index].round(0).values,
    "z_score": anom.round(2).values,
    "type": np.where(anom.values > 0, "unusual HIGH", "unusual LOW"),
})
out

In [ ]:
# chart: 2025 demand with anomalies marked
plt.figure(figsize=(10, 3.8))
y_test.plot(color="#334155", label="2025 demand")
best_pred.plot(color="#0D9488", alpha=0.7, label="model expected")
plt.scatter(anom.index, y_test[anom.index], color="#DC2626", zorder=5, label="anomaly")
plt.title(f"Anomaly detection on 2025 test data (|z| > 3)")
plt.ylabel("Demand (MW)")
plt.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Save outputs for the web app

In [ ]:
import json
import joblib

# output folder: Kaggle working dir or local "output"
OUT = "/kaggle/working" if os.path.isdir("/kaggle") else "output"
os.makedirs(OUT, exist_ok=True)

# 1) full daily series
df.to_csv(f"{OUT}/daily.csv", index=False)

# 2) model comparison
with open(f"{OUT}/models.json", "w") as f:
    json.dump({"best_model": best_name,
               "test_period": "2025-01-01 to 2025-10-25",
               "results": rows}, f, indent=2)

# 3) next month forecast
with open(f"{OUT}/next_month.json", "w") as f:
    json.dump({
        "month": "2025-11",
        "month_name": "November 2025",
        "days": [{"date": str(ts.date()), "demand_mw": round(v, 1)}
                 for ts, v in fmonth.items()],
        "stats": {"total_gwh": round(total_gwh, 1), "avg_mw": round(avg_mw, 1),
                  "peak_mw": round(fmonth.max(), 1), "peak_date": str(peak_day.date()),
                  "last_year_gwh": round(last_nov.sum() * 24 / 1000, 1),
                  "yoy_pct": round(yoy, 1), "rmse_mw": round(rmse, 1)},
    }, f, indent=2)

# 4) test predictions with anomaly flags
pd.DataFrame({"date": X_test.index, "actual_mw": y_test.values,
              "predicted_mw": best_pred.round(1).values, "z": z.round(3).values,
              "is_anomaly": (abs(z) > 3).astype(int).values}
             ).to_csv(f"{OUT}/test_predictions.csv", index=False)

# 5) best model
joblib.dump(best_model, f"{OUT}/best_model.joblib")

print("saved to", OUT, ":", sorted(os.listdir(OUT)))